### Using the TimeDB REST API

In [1]:
try:
    import google.colab
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/rebase-energy/timedb/main/examples/colab_setup.py',
        '/tmp/colab_setup.py'
    )
    exec(open('/tmp/colab_setup.py').read())
except ImportError:
    pass  # Not running in Google Colab


In [2]:
import io
import json
from datetime import datetime, timezone, timedelta

import polars as pl
import requests
import timedb as td

API_BASE_URL = "http://127.0.0.1:8000"
headers = {"Content-Type": "application/json"}
base_time = datetime(2025, 1, 1, tzinfo=timezone.utc)

## Part 1: Setup

Create the database schema via SDK (admin task — the API cannot create/delete schemas).

In [3]:
td.delete()
td.create()

Creating database schema...
  ✓ PostgreSQL: series
  ✓ ClickHouse: runs, flat, overlapping_short/medium/long
✓ Schema created successfully


## Part 2: Start the API Server

Start the server before making API calls. In a notebook we run it as a background process.

In [4]:
import subprocess, time

subprocess.run(["pkill", "-f", "uvicorn.*timedb"], capture_output=True)
time.sleep(1)

process = subprocess.Popen(
    ["timedb", "api", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)

response = requests.get(f"{API_BASE_URL}/")
print(f"✓ {response.json()['name']} v{response.json().get('version')} running")

✓ TimeDB API v0.1.4 running


## Part 3: Insert Data Using the API

Now let's create some sample time series data and insert it using the REST API.

In [5]:
for cfg in [
    {"name": "wind_speed", "unit": "m/s", "labels": {"site": "Gotland"}},
    {"name": "power_forecast", "unit": "MW", "labels": {"site": "Gotland"}, "overlapping": True},
]:
    r = requests.post(f"{API_BASE_URL}/series", json=cfg, headers=headers)
    r.raise_for_status()
    print(f"✓ {cfg['name']}: series_id={r.json()['series_id']}")

✓ wind_speed: series_id=1
✓ power_forecast: series_id=2


### POST /series/many — Bulk Create

Create multiple series in a single round-trip. Useful when setting up many turbines or sites at once.

In [6]:
payload = {
    "series": [
        {"name": "wind_power", "unit": "MW", "labels": {"turbine": f"T0{i}"}, "overlapping": True}
        for i in range(1, 4)
    ]
}
r = requests.post(f"{API_BASE_URL}/series/many", json=payload, headers=headers)
r.raise_for_status()
print(r.json())  # {"series_ids": [3, 4, 5]}

{'series_ids': [3, 4, 5]}


### 3.1: POST /values — JSON

In [7]:
response = requests.post(
    f"{API_BASE_URL}/values",
    json={
        "name": "wind_speed",
        "labels": {"site": "Gotland"},
        "data": [
            {"valid_time": (base_time + timedelta(hours=i)).isoformat(), "value": 20.0 + i * 0.3}
            for i in range(24)
        ],
    },
    headers=headers,
)
response.raise_for_status()
print(f"✓ wind_speed: {response.json()}")

response = requests.post(
    f"{API_BASE_URL}/values",
    json={
        "name": "power_forecast",
        "labels": {"site": "Gotland"},
        "data": [
            {"valid_time": (base_time + timedelta(hours=i)).isoformat(), "value": 60.0 - i * 0.5}
            for i in range(24)
        ],
    },
    headers=headers,
)
response.raise_for_status()
print(f"✓ power_forecast: {response.json()}")

✓ wind_speed: {'run_id': '019d2dde-b85c-717e-8665-3e615e87df9a', 'series_id': 1, 'rows_inserted': 24}


✓ power_forecast: {'run_id': '019d2dde-b894-71ff-adca-5a614b6d4944', 'series_id': 2, 'rows_inserted': 24}


### 3.2: POST /values — Arrow IPC

Send a binary Arrow IPC stream. The series is identified via query parameters.

In [8]:
df = pl.DataFrame({
    "valid_time": pl.Series([base_time + timedelta(days=1, hours=i) for i in range(6)]).cast(pl.Datetime("us", "UTC")),
    "value": [21.0 + i * 0.4 for i in range(6)],
})

buf = io.BytesIO()
df.write_ipc_stream(buf)

response = requests.post(
    f"{API_BASE_URL}/values",
    params={"name": "wind_speed", "labels": json.dumps({"site": "Gotland"})},
    data=buf.getvalue(),
    headers={"Content-Type": "application/vnd.apache.arrow.stream"},
)
response.raise_for_status()
print(f"✓ Arrow IPC insert: {response.json()}")

✓ Arrow IPC insert: {'run_id': '019d2dde-b8c0-710b-a98c-093ba307ea67', 'series_id': 1, 'rows_inserted': 6}


### 3.3: GET /series — List and Filter

In [9]:
r = requests.get(f"{API_BASE_URL}/series", headers=headers)
r.raise_for_status()
for s in r.json():
    print(f"  series_id={s['series_id']}: {s['name']} ({s['unit']}) labels={s['labels']}")

r = requests.get(f"{API_BASE_URL}/series/labels", params={"label_key": "site"}, headers=headers)
print(r.json())

r = requests.get(f"{API_BASE_URL}/series/count", headers=headers)
print(f"count: {r.json()['count']}")

  series_id=2: power_forecast (MW) labels={'site': 'Gotland'}
  series_id=3: wind_power (MW) labels={'turbine': 'T01'}
  series_id=4: wind_power (MW) labels={'turbine': 'T02'}
  series_id=5: wind_power (MW) labels={'turbine': 'T03'}
  series_id=1: wind_speed (m/s) labels={'site': 'Gotland'}
{'label_key': 'site', 'values': ['Gotland']}
count: 5


## Part 4: Read Data

In [10]:
response = requests.get(
    f"{API_BASE_URL}/values",
    params={
        "name": "wind_speed",
        "labels": json.dumps({"site": "Gotland"}),
        "start_valid": base_time.isoformat(),
        "end_valid": (base_time + timedelta(hours=24)).isoformat(),
    },
    headers=headers,
)
response.raise_for_status()
data = response.json()
print(f"✓ {data['count']} records")
print(pl.DataFrame(data["data"]).head())

✓ 24 records
shape: (5, 2)
┌──────────────────────────┬───────┐
│ valid_time               ┆ value │
│ ---                      ┆ ---   │
│ str                      ┆ f64   │
╞══════════════════════════╪═══════╡
│ 2025-01-01T01:00:00+0000 ┆ 20.3  │
│ 2025-01-01T02:00:00+0000 ┆ 20.6  │
│ 2025-01-01T03:00:00+0000 ┆ 20.9  │
│ 2025-01-01T04:00:00+0000 ┆ 21.2  │
│ 2025-01-01T05:00:00+0000 ┆ 21.5  │
└──────────────────────────┴───────┘


### 4.1: GET /values — overlapping=true

Returns one row per `(knowledge_time, valid_time)` — the full forecast history across all runs.

In [11]:
response = requests.get(
    f"{API_BASE_URL}/values",
    params={
        "name": "power_forecast",
        "start_valid": base_time.isoformat(),
        "end_valid": (base_time + timedelta(hours=6)).isoformat(),
        "overlapping": "true",
    },
    headers=headers,
)
response.raise_for_status()
data = response.json()
print(f"✓ {data['count']} records (overlapping=true)")
print(pl.DataFrame(data["data"]).head())

✓ 6 records (overlapping=true)
shape: (5, 3)
┌──────────────────────────┬──────────────────────────┬───────┐
│ knowledge_time           ┆ valid_time               ┆ value │
│ ---                      ┆ ---                      ┆ ---   │
│ str                      ┆ str                      ┆ f64   │
╞══════════════════════════╪══════════════════════════╪═══════╡
│ 2026-03-27T05:57:44+0000 ┆ 2025-01-01T01:00:00+0000 ┆ 59.5  │
│ 2026-03-27T05:57:44+0000 ┆ 2025-01-01T02:00:00+0000 ┆ 59.0  │
│ 2026-03-27T05:57:44+0000 ┆ 2025-01-01T03:00:00+0000 ┆ 58.5  │
│ 2026-03-27T05:57:44+0000 ┆ 2025-01-01T04:00:00+0000 ┆ 58.0  │
│ 2026-03-27T05:57:44+0000 ┆ 2025-01-01T05:00:00+0000 ┆ 57.5  │
└──────────────────────────┴──────────────────────────┴───────┘


### 4.2: GET /values — Arrow IPC

Set `Accept: application/vnd.apache.arrow.stream` to receive a typed binary stream.

In [12]:
response = requests.get(
    f"{API_BASE_URL}/values",
    params={"name": "wind_speed", "labels": json.dumps({"site": "Gotland"})},
    headers={"Accept": "application/vnd.apache.arrow.stream"},
)
response.raise_for_status()

df = pl.read_ipc_stream(response.content)
print(f"✓ {len(df)} rows, schema: {df.schema}")
print(df.head())

✓ 30 rows, schema: Schema({'valid_time': Datetime(time_unit='us', time_zone='UTC'), 'value': Float64})


shape: (5, 2)
┌─────────────────────────┬───────┐
│ valid_time              ┆ value │
│ ---                     ┆ ---   │
│ datetime[μs, UTC]       ┆ f64   │
╞═════════════════════════╪═══════╡
│ 2025-01-01 00:00:00 UTC ┆ 20.0  │
│ 2025-01-01 01:00:00 UTC ┆ 20.3  │
│ 2025-01-01 02:00:00 UTC ┆ 20.6  │
│ 2025-01-01 03:00:00 UTC ┆ 20.9  │
│ 2025-01-01 04:00:00 UTC ┆ 21.2  │
└─────────────────────────┴───────┘


## Part 5: POST /write — Multi-Series Insert

Insert data for multiple series in one call from a long/tidy-format table.
All target series must already exist. Series routing is via `name_col` and `label_cols` query params.

In [13]:
for zone in ["A", "B"]:
    r = requests.post(
        f"{API_BASE_URL}/series",
        json={"name": "temperature", "unit": "degC", "labels": {"zone": zone}},
        headers=headers,
    )
    r.raise_for_status()
    print(f"✓ temperature zone={zone}: series_id={r.json()['series_id']}")

✓ temperature zone=A: series_id=6
✓ temperature zone=B: series_id=7


### 5.1: POST /write — JSON

In [14]:
write_dates = [base_time + timedelta(hours=i) for i in range(6)]

records = [
    {"name": "temperature", "zone": zone, "valid_time": dt.isoformat(), "value": base + i * 0.2}
    for zone, base in [("A", 22.0), ("B", 18.0)]
    for i, dt in enumerate(write_dates)
]

response = requests.post(
    f"{API_BASE_URL}/write",
    params={"name_col": "name", "label_cols": "zone"},
    json=records,
    headers=headers,
)
response.raise_for_status()
for r in response.json():
    print(f"  series_id={r['series_id']}, run_id={r['run_id']}")

  series_id=7, run_id=019d2dde-b9a4-76a8-b07c-f7e52fd4de03
  series_id=6, run_id=019d2dde-b9a4-76a8-b07c-f7e52fd4de03


### 5.2: POST /write — Arrow IPC

In [15]:
df_write = pl.DataFrame({
    "name": ["temperature"] * 12,
    "zone": ["A"] * 6 + ["B"] * 6,
    "valid_time": pl.Series(write_dates + write_dates).cast(pl.Datetime("us", "UTC")),
    "value": [22.0 + i * 0.2 for i in range(6)] + [18.0 + i * 0.2 for i in range(6)],
})

buf = io.BytesIO()
df_write.write_ipc_stream(buf)

response = requests.post(
    f"{API_BASE_URL}/write",
    params={"name_col": "name", "label_cols": "zone"},
    data=buf.getvalue(),
    headers={"Content-Type": "application/vnd.apache.arrow.stream"},
)
response.raise_for_status()
for r in response.json():
    print(f"  series_id={r['series_id']}, run_id={r['run_id']}")

  series_id=6, run_id=019d2dde-b9cf-71c2-8233-7a98117ab94e
  series_id=7, run_id=019d2dde-b9cf-71c2-8233-7a98117ab94e


## Summary

| Endpoint | Description |
|----------|-------------|
| `POST /series` | Create a time series (get-or-create) |
| `POST /series/many` | Bulk create multiple series |
| `POST /values` | Insert single-series data — JSON or Arrow IPC (`Content-Type`) |
| `POST /write` | Insert multi-series long-format data — JSON or Arrow IPC |
| `GET /values` | Read values — filter by name/labels/time range; JSON or Arrow IPC (`Accept`) |
| `GET /series` | List/filter series |
| `GET /series/labels` | List unique label values for a key |
| `GET /series/count` | Count matching series |